In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

FEATURE_COLS = ["open", "high", "low", "close", "bid_price1", "ask_price1",
                "volume", "amount", "bid_volume1", "ask_volume1"]
PRICE_COLS   = ["open", "high", "low", "close", "bid_price1", "ask_price1"]
VOL_COLS     = ["volume", "amount", "bid_volume1", "ask_volume1"]
OHLC_COLS    = ["open", "high", "low", "close"]
SCALE_FIELDS = ["open", "high", "low", "close", "amount", "bid_price1", "ask_price1"]
SEQ_LEN      = 40
MODEL_PATH   = "transformer_model_30min3.json"


class StockTransformer(nn.Module):
    def __init__(self, n_feat, d_model, nhead, nlayers, dim_ff, seq_len):
        super().__init__()
        self.proj = nn.Linear(n_feat, d_model)
        self.pos  = nn.Parameter(torch.zeros(1, seq_len, d_model))
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, 0.1,
                                           batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, nlayers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x):
        return self.head(self.encoder(self.proj(x) + self.pos).mean(1)).squeeze(-1)


def to_canonical(df: pd.DataFrame, is_local: bool = False) -> pd.DataFrame:
    df = df.copy()
    
    if "instrument" in df.columns and "instrument_id" not in df.columns:
        df.rename(columns={"instrument": "instrument_id"}, inplace=True)
    
    if not pd.api.types.is_datetime64_any_dtype(df["date"]):
        df["date"] = pd.to_datetime(df["date"])
    
    if is_local:
        for c in OHLC_COLS:
            df.loc[df[c] == -1, c] = np.nan
        for c in SCALE_FIELDS:
            df[c] = df[c] / 100.0
    
    for c in VOL_COLS:
        if c in df.columns:
            df[c] = np.log1p(df[c].clip(lower=0))
    
    df.sort_values(["instrument_id", "date"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    for c in OHLC_COLS:
        if c in df.columns:
            df[c] = df.groupby("instrument_id")[c].ffill()
    
    return df


def build_windows(df: pd.DataFrame, start_date: str, end_date: str, 
                  mode: str, stats: tuple = None):
    sd = pd.Timestamp(start_date)
    ed = pd.Timestamp(end_date)
    mean, std = stats if stats else (None, None)
    
    wins, ys, meta_rows = [], [], []
    
    for key, sub in df.groupby("instrument_id", sort=False):
        if len(sub) <= SEQ_LEN:
            continue
        
        feats = sub[FEATURE_COLS].to_numpy(np.float32)
        day   = sub["date"].dt.normalize().to_numpy()
        eod   = np.flatnonzero(np.append(day[1:] != day[:-1], True))
        
        for j, p in enumerate(eod):
            d = pd.Timestamp(day[eod][j])
            if p + 1 < SEQ_LEN or d < sd or d > ed:
                continue
            
            win = feats[p - SEQ_LEN + 1: p + 1]
            if not np.isfinite(win).all():
                continue
                
            wins.append(win)
            meta_rows.append({"date": d, "key": key})
            
            if mode == "train" and j + 1 < len(eod):
                cpx = sub["close"].to_numpy(np.float64)[eod]
                if cpx[j] > 0:
                    r = cpx[j + 1] / cpx[j] - 1.0
                    ys.append(np.float32(r) if np.isfinite(r) else np.nan)
    
    X = np.stack(wins).astype(np.float32)
    idx_df = pd.DataFrame(meta_rows)
    
    if mode == "train":
        mean = X.reshape(-1, len(FEATURE_COLS)).mean(0).astype(np.float32)
        std  = X.reshape(-1, len(FEATURE_COLS)).std(0).astype(np.float32) + 1e-6
    
    X = ((X - mean) / std).astype(np.float32)
    
    y = None
    if mode == "train" and ys:
        y = np.array(ys, np.float32)
        p1, p99 = np.percentile(y[~np.isnan(y)], [1, 99])
        y = np.clip(y, p1, p99).astype(np.float32)
    
    return X, y, idx_df, (mean, std)


def load_model(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        ckpt = json.load(f)
    
    state_dict = {}
    for k, v in ckpt["state_dict"].items():
        dtype = getattr(torch, v["dtype"])
        state_dict[k] = torch.tensor(v["data"], dtype=dtype).reshape(v["shape"])
    
    ckpt["state_dict"] = state_dict
    return ckpt


def main(datasources: dict, start_date: str, end_date: str) -> pd.DataFrame:
    import dai
    
    ckpt = load_model(MODEL_PATH)
    stats = (np.asarray(ckpt["mean"], dtype=np.float32), 
             np.asarray(ckpt["std"], dtype=np.float32))
    
    model = StockTransformer(**ckpt["model_cfg"])
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    model.to(device)
    
    table = datasources.get("bar30m") or next(iter(datasources.values()))
    
    buf = (pd.Timestamp(start_date) - pd.Timedelta(days=20)).strftime("%Y-%m-%d")
    
    raw = dai.query(
        f"SELECT date, instrument, {','.join(FEATURE_COLS)} FROM {table} "
        f"WHERE date >= '{buf}' AND date <= '{end_date}' "
        f"ORDER BY instrument, date"
    ).df()
    
    canon = to_canonical(raw, is_local=False)
    
    Xte, _, idx_df, _ = build_windows(canon, start_date, end_date, 
                                       mode="infer", stats=stats)
    
    BATCH_SIZE = 1024
    scores = []
    with torch.no_grad():
        for i in range(0, len(Xte), BATCH_SIZE):
            xb = torch.from_numpy(Xte[i:i+BATCH_SIZE]).to(device)
            out = model(xb).cpu().numpy()
            scores.append(out)
    
    scores = np.concatenate(scores).astype(np.float32)
    
    result = idx_df.copy()
    result["score"] = scores
    result.rename(columns={"key": "instrument"}, inplace=True)
    
    return result[["date", "instrument", "score"]]